# $\mathbb{Z}_2^F \times \mathbb{Z}_2^T$ - Majorana Hamiltonians

Created: 31-07-2026

Iterate on [previous notebook](z2_f_x_z2_t_hamiltonians.ipynb)

# Imports

In [1]:
from time import time

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from itertools import product

In [6]:
import quimb.tensor as qtn
import quimb as qu

In [7]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

In [8]:
from humanize import naturalsize

# Definitions

In [9]:
group_quads = list(product([0,1], repeat=4))

In [10]:
spin_ops_dict = {
    (0,0): [("I", 1), ("y", 1)],
    (1,1): [("I", 1), ("y", -1)],
    (0,1): [("z", 1), ("x", 1j)],
    (1,0): [("z", 1), ("x", -1j)]
}

In [11]:
majorana_1 = [('+', 1) ,('-', 1)]
majorana_2 = [('+', 1j),  ('-', -1j)]

In [60]:
def get_triv_terms(L, strength_scaling=1):
    terms = [
        ['z|', [[-1*strength_scaling, i] for i in range(L)]],
        ['|n', [[strength_scaling, i] for i in range(L)]],
        ['|I', [[-1*strength_scaling, i] for i in range(L)]],
    ]

    return terms

In [88]:
def get_triv_cocycle_nontriv_fermion_decoration_terms(L, strength_scaling=1):
    ss = strength_scaling

    spin_terms=[
        [f'z|{op_l}{op_r}', [[-1j*s_l*s_r*ss, i, (i-1)%L, i] for i in range(L)]]
        for op_l, s_l in majorana_2
        for op_r, s_r in majorana_1
    ]

    fermion_terms = [
        [f'yy|I', [[-1*ss, i, (i+1)%L, i] for i in range(L)]],
        [f'yy|n', [[2*ss, i, (i+1)%L, i] for i in range(L)]]
    ]

    terms = spin_terms + fermion_terms

    return terms

In [14]:
def cocycle_phase(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    exponent = (
        (g_in - g_left)*(g_right - g_in)
        - (g_out - g_left)*(g_right - g_out)
    )
    exponent = exponent % 2

    # i.e. (-1)**exponent
    out = -1 if exponent else 1

    return out

In [15]:
majorana_hopping_terms = [
    (f'{op_l}{op_r}', -1j*s_l*s_r, (0, 1))
    for op_l, s_l in majorana_2
    for op_r, s_r in majorana_1
]

In [16]:
majorana_hopping_terms

[('++', (1-0j), (0, 1)),
 ('+-', (1-0j), (0, 1)),
 ('-+', (-1+0j), (0, 1)),
 ('--', (-1+0j), (0, 1))]

In [22]:
def get_nontriv_group_quad_terms(group_quad, L, strength_scaling=1):
    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength

        # Cases where we have majorana hopping or not
        if (g_in + g_out)%2:
            for ferm_op_string, ferm_strength, ferm_indices in majorana_hopping_terms:
                op_string = f"{spin_string}|{ferm_op_string}"
    
                base_index = [0, 1, 2, *ferm_indices]
                all_indices = [
                    [(x+i)%L for x in base_index]
                    for i in range(L)
                ]
        
                phase = cocycle_phase(group_quad)
                scaling_constant = (
                    strength
                    *strength_scaling
                    *cocycle_phase(group_quad)
                    *ferm_strength
                )
                current_term = [
                    op_string, [[scaling_constant, *indices] for indices in all_indices]
                ]
        
                terms.append(current_term)
        else:
            op_string = f"{spin_string}|"

            base_index = [0, 1, 2]
            all_indices = [
                [(x+i)%L for x in base_index]
                for i in range(L)
            ]
    
            phase = cocycle_phase(group_quad)
            scaling_constant = (
                strength
                *strength_scaling
                *cocycle_phase(group_quad)
            )
            current_term = [
                op_string, [[scaling_constant, *indices] for indices in all_indices]
            ]
    
            terms.append(current_term)

    return terms

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [18]:
def get_nontriv_cocycle_nontriv_fermion_decoration_terms(L, strength_scaling=1):
    spin_terms = [
        l for group_quad in group_quads
        for l in get_nontriv_group_quad_terms(group_quad, L, strength_scaling)
    ]

    fermion_terms = [
        [f'yy|I', [[-1, i, (i+1)%L, i] for i in range(L)]],
        [f'yy|n', [[2, i, (i+1)%L, i] for i in range(L)]]
    ]

    terms = spin_terms + fermion_terms

    return terms

In [55]:
def get_triv_to_n1_non_triv_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = get_triv_terms(L, 1-t)

    non_triv_terms = get_triv_cocycle_nontriv_fermion_decoration_terms(
        L,
        t
    )

    all_terms = triv_terms + non_triv_terms

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

In [110]:
def get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = get_triv_cocycle_nontriv_fermion_decoration_terms(
        L,
        1-t
    )

    non_triv_spin_terms = [
        l for group_quad in group_quads
        for l in get_nontriv_group_quad_terms(group_quad, L, t)
    ]

    non_triv_fermion_terms = [
        [f'yy|I', [[-1*t, i, (i+1)%L, i] for i in range(L)]],
        [f'yy|n', [[2*t, i, (i+1)%L, i] for i in range(L)]]
    ]

    all_terms = (
        triv_terms
        + non_triv_spin_terms
        + non_triv_fermion_terms
    )

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

Test energies:

In [70]:
L=4

In [99]:
parameters = np.linspace(0, 1, 21)

In [100]:
triv_energies = list()

L=4
spin_basis = spin_basis_1d(L)
fermion_basis = spinless_fermion_basis_1d(L)
basis = tensor_basis(spin_basis, fermion_basis)

for t in parameters:
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    triv_energies.append(e)

/tmp/ipykernel_22190/2905373412.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [101]:
triv_energies

[array([-8.]),
 array([-7.60788297]),
 array([-7.23311684]),
 array([-6.87816503]),
 array([-6.54557933]),
 array([-6.23808388]),
 array([-5.9588872]),
 array([-5.73173096]),
 array([-5.64222051]),
 array([-5.5760182]),
 array([-5.54999277]),
 array([-5.71879951]),
 array([-5.91157736]),
 array([-6.12485309]),
 array([-6.35584424]),
 array([-6.60218762]),
 array([-6.8618152]),
 array([-7.13290458]),
 array([-7.41385802]),
 array([-7.70328782]),
 array([-8.])]

In [111]:
non_triv_energies = list()

L=4
spin_basis = spin_basis_1d(L)
fermion_basis = spinless_fermion_basis_1d(L)
basis = tensor_basis(spin_basis, fermion_basis)

for t in parameters:
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    non_triv_energies.append(e)

/tmp/ipykernel_22190/662957289.py:27: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [112]:
non_triv_energies

[array([-8.]),
 array([-7.90131556]),
 array([-7.80555128]),
 array([-7.71320963]),
 array([-7.6249031]),
 array([-7.54138127]),
 array([-7.46356421]),
 array([-7.3925824]),
 array([-7.32982213]),
 array([-7.27697286]),
 array([-7.23606798]),
 array([-7.20950231]),
 array([-7.2]),
 array([-7.21049732]),
 array([-7.24390889]),
 array([-7.30277564]),
 array([-7.38885438]),
 array([-7.50277564]),
 array([-7.64390889]),
 array([-7.81049732]),
 array([-8.])]

# Sweep

In [113]:
parameters = np.linspace(0, 1, 21)

## 4 site

In [114]:
L = 4

## Trivial cocycle

In [115]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_triv_to_nontriv_n1_4_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/2905373412.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:01<00:00, 15.07it/s]


## Nontrivial cocycle

In [116]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_nontriv_n1_to_nontriv_cocyle_4_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/662957289.py:27: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 21.45it/s]


## 8 site

In [117]:
L = 8

## Trivial cocycle

In [118]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_triv_to_nontriv_n1_8_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/2905373412.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:23<00:00,  1.10s/it]


## Nontrivial cocycle

In [119]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_nontriv_n1_to_nontriv_cocyle_8_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/662957289.py:27: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:21<00:00,  1.02s/it]


## 10 site

In [120]:
L = 10

## Trivial cocycle

In [121]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_triv_to_nontriv_n1_10_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/2905373412.py:15: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [17:40<00:00, 50.50s/it]


## Nontrivial cocycle

In [122]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_majorana_nontriv_n1_to_nontriv_cocyle_10_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                     | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_22190/662957289.py:27: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [06:52<00:00, 19.67s/it]
